# Task 2: Automated Unstructured Web Scraping & Data Extraction Pipeline

**Objective:** Build an automated pipeline that harvests data records from a remote web environment (quotes.toscrape.com), extracts target parameters, cleans the data, and writes it to a structured CSV / SQLite database.

**Source:** http://quotes.toscrape.com — a public sandbox site built specifically for practicing web scraping (safe, legal, no rate-limit issues).

**Tools:** `requests`, `BeautifulSoup`, `pandas`, `sqlite3`

**Pipeline stages:**
1. Setup & imports
2. Fetch raw HTML across all paginated pages (automated crawl)
3. Parse unstructured listings & extract core parameters (quote text, author, tags)
4. Clean whitespace / artifacts
5. Structure into a DataFrame
6. Write to CSV
7. Write to a SQLite database file
8. Quick validation / sanity checks


## 1. Setup & Imports

In [1]:
!pip install requests beautifulsoup4 pandas -q

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
import time
import re
from urllib.parse import urljoin

BASE_URL = "http://quotes.toscrape.com"
HEADERS = {"User-Agent": "Mozilla/5.0 (Task2-Scraper-Educational/1.0)"}


## 2. Automated Crawl Across Pages

The site paginates quotes (`/page/1/`, `/page/2/`, ...). We crawl until there is no "Next" button, collecting raw HTML for each page. A small delay is added between requests to be a polite, responsible scraper.

In [3]:
def fetch_all_pages(base_url, delay=0.5, max_pages=50):
    """
    Crawl quotes.toscrape.com page by page, following the 'Next' link,
    and return a list of raw HTML strings (one per page).
    """
    pages_html = []
    url = base_url + "/"
    page_count = 0

    while url and page_count < max_pages:
        resp = requests.get(url, headers=HEADERS, timeout=10)
        resp.raise_for_status()
        pages_html.append(resp.text)
        page_count += 1

        soup = BeautifulSoup(resp.text, "html.parser")
        next_btn = soup.select_one("li.next > a")
        url = urljoin(base_url, next_btn["href"]) if next_btn else None

        time.sleep(delay)  # be polite to the server

    print(f"Fetched {page_count} pages.")
    return pages_html

raw_pages = fetch_all_pages(BASE_URL)


Fetched 10 pages.


## 3. Parse Listings & Extract Core Parameters

For each quote "card" on a page we extract:
- `quote_text` — the quote itself
- `author` — author name
- `tags` — list of tags, joined into a single string
- `author_url` — link to the author's bio page (extra structured field)


In [4]:
def parse_page(html):
    """
    Parse a single page's HTML and return a list of dicts,
    one per quote listing found on that page.
    """
    soup = BeautifulSoup(html, "html.parser")
    records = []

    for card in soup.select("div.quote"):
        text_el = card.select_one("span.text")
        author_el = card.select_one("small.author")
        author_link_el = card.select_one("span > a")
        tag_els = card.select("div.tags a.tag")

        record = {
            "quote_text": text_el.get_text(strip=True) if text_el else None,
            "author": author_el.get_text(strip=True) if author_el else None,
            "author_url": urljoin(BASE_URL, author_link_el["href"]) if author_link_el else None,
            "tags": ", ".join(t.get_text(strip=True) for t in tag_els) if tag_els else "",
        }
        records.append(record)

    return records

all_records = []
for html in raw_pages:
    all_records.extend(parse_page(html))

print(f"Extracted {len(all_records)} raw records.")
all_records[:3]


Extracted 100 raw records.


[{'quote_text': '“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”',
  'author': 'Albert Einstein',
  'author_url': 'http://quotes.toscrape.com/author/Albert-Einstein',
  'tags': 'change, deep-thoughts, thinking, world'},
 {'quote_text': '“It is our choices, Harry, that show what we truly are, far more than our abilities.”',
  'author': 'J.K. Rowling',
  'author_url': 'http://quotes.toscrape.com/author/J-K-Rowling',
  'tags': 'abilities, choices'},
 {'quote_text': '“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”',
  'author': 'Albert Einstein',
  'author_url': 'http://quotes.toscrape.com/author/Albert-Einstein',
  'tags': 'inspirational, life, live, miracle, miracles'}]

## 4. Clean Whitespace Artifacts & Normalize Text

Raw scraped text from `quotes.toscrape.com` wraps each quote in curly “smart quotes” (`“...”`) and may contain stray whitespace/newlines. We strip these artifacts and normalize the text fields.

In [5]:
def clean_text(value):
    if value is None:
        return None
    value = value.strip()
    # remove the leading/trailing curly quote characters used by the site
    value = value.strip("\u201c\u201d")
    # collapse internal whitespace/newlines into single spaces
    value = re.sub(r"\s+", " ", value)
    return value.strip()

def clean_record(record):
    return {
        "quote_text": clean_text(record["quote_text"]),
        "author": clean_text(record["author"]),
        "author_url": record["author_url"],
        "tags": clean_text(record["tags"]),
    }

cleaned_records = [clean_record(r) for r in all_records]
cleaned_records[:3]


[{'quote_text': 'The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.',
  'author': 'Albert Einstein',
  'author_url': 'http://quotes.toscrape.com/author/Albert-Einstein',
  'tags': 'change, deep-thoughts, thinking, world'},
 {'quote_text': 'It is our choices, Harry, that show what we truly are, far more than our abilities.',
  'author': 'J.K. Rowling',
  'author_url': 'http://quotes.toscrape.com/author/J-K-Rowling',
  'tags': 'abilities, choices'},
 {'quote_text': 'There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.',
  'author': 'Albert Einstein',
  'author_url': 'http://quotes.toscrape.com/author/Albert-Einstein',
  'tags': 'inspirational, life, live, miracle, miracles'}]

## 5. Structure into a DataFrame

In [6]:
df = pd.DataFrame(cleaned_records)

# drop exact duplicate rows and any rows missing the core quote text
df = df.drop_duplicates().dropna(subset=["quote_text"]).reset_index(drop=True)

print(df.shape)
df.head(10)


(100, 4)


,quote_text,author,author_url,tags
0,The world as we have created it is a process o...,Albert Einstein,http://quotes.toscrape.com/author/Albert-Einstein,"change, deep-thoughts, thinking, world"
1,"It is our choices, Harry, that show what we tr...",J.K. Rowling,http://quotes.toscrape.com/author/J-K-Rowling,"abilities, choices"
2,There are only two ways to live your life. One...,Albert Einstein,http://quotes.toscrape.com/author/Albert-Einstein,"inspirational, life, live, miracle, miracles"
3,"The person, be it gentleman or lady, who has n...",Jane Austen,http://quotes.toscrape.com/author/Jane-Austen,"aliteracy, books, classic, humor"
4,"Imperfection is beauty, madness is genius and ...",Marilyn Monroe,http://quotes.toscrape.com/author/Marilyn-Monroe,"be-yourself, inspirational"
5,Try not to become a man of success. Rather bec...,Albert Einstein,http://quotes.toscrape.com/author/Albert-Einstein,"adulthood, success, value"
6,It is better to be hated for what you are than...,André Gide,http://quotes.toscrape.com/author/Andre-Gide,"life, love"
7,"I have not failed. I've just found 10,000 ways...",Thomas A. Edison,http://quotes.toscrape.com/author/Thomas-A-Edison,"edison, failure, inspirational, paraphrased"
8,A woman is like a tea bag; you never know how ...,Eleanor Roosevelt,http://quotes.toscrape.com/author/Eleanor-Roos...,misattributed-eleanor-roosevelt
9,"A day without sunshine is like, you know, night.",Steve Martin,http://quotes.toscrape.com/author/Steve-Martin,"humor, obvious, simile"


## 6. Write Output to a Structured CSV

In [7]:
csv_path = "quotes_dataset.csv"
df.to_csv(csv_path, index=False, encoding="utf-8")
print(f"Saved {len(df)} rows to {csv_path}")


Saved 100 rows to quotes_dataset.csv


## 7. Write Output to a SQLite Database File\n\n(Satisfies the \"database file or structured CSV\" requirement — we do both for completeness.)

In [8]:
db_path = "quotes_dataset.db"
conn = sqlite3.connect(db_path)
df.to_sql("quotes", conn, if_exists="replace", index=False)
conn.close()
print(f"Saved {len(df)} rows to table 'quotes' in {db_path}")


Saved 100 rows to table 'quotes' in quotes_dataset.db


## 8. Validation / Sanity Checks

In [9]:
# Reload from CSV and from DB to confirm everything was written correctly
df_csv_check = pd.read_csv(csv_path)
conn = sqlite3.connect(db_path)
df_db_check = pd.read_sql("SELECT * FROM quotes", conn)
conn.close()

print("CSV rows:", len(df_csv_check))
print("DB rows :", len(df_db_check))
print("\nTop authors by quote count:")
print(df["author"].value_counts().head())

assert len(df_csv_check) == len(df_db_check) == len(df), "Row counts mismatch!"
print("\nAll checks passed.")


CSV rows: 100
DB rows : 100

Top authors by quote count:
author
Albert Einstein    10
J.K. Rowling        9
Marilyn Monroe      7
Dr. Seuss           6
Mark Twain          6
Name: count, dtype: int64

All checks passed.


## Summary

- Crawled all paginated pages of `quotes.toscrape.com` automatically (following the "Next" link).
- Extracted core parameters: quote text, author, author URL, tags.
- Cleaned whitespace artifacts and smart-quote characters.
- Structured the data into a pandas DataFrame and removed duplicates.
- Persisted the final dataset to both `quotes_dataset.csv` and `quotes_dataset.db` (SQLite).

This notebook can be adapted to other public sources by swapping the `parse_page` selectors to match the target site's HTML structure.